# SARIMAX with Exogenous Variables

This notebook demonstrates the practical power of adding exogenous variables
to a SARIMA model.  The star example comes from Portilla's course (Section
06-07): predicting **restaurant visitors** using a **holiday indicator** as
an exogenous variable.

The hypothesis is simple — people eat out more on holidays.  By telling
the model *when* holidays occur, we should get better forecasts than a
pure SARIMA that has no knowledge of the calendar.

This notebook covers:
1. The restaurant visitors dataset and holiday variable
2. Baseline SARIMA (no exogenous)
3. SARIMAX with holiday exogenous — and the improvement
4. Feature engineering for exogenous variables
5. Guidelines for choosing exogenous variables
6. Application to Alcohol Sales with a trend exogenous

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. Load Restaurant Visitors Data

This dataset (from Portilla's course) contains **daily** restaurant visitor
counts for multiple restaurants, along with a **holiday indicator** and the
holiday name.

In [ ]:
rest = pd.read_csv(
    DATA_DIR / "RestaurantVisitors.csv",
    parse_dates=["date"],
    index_col="date",
)

print(f"Shape: {rest.shape}")
print(f"Date range: {rest.index[0].date()} to {rest.index[-1].date()}")
print(f"Columns: {list(rest.columns)}")
rest.head(10)

In [ ]:
# Explore the holiday column
print(f"Holiday distribution:")
print(rest["holiday"].value_counts())
print(f"\nTotal holidays: {rest['holiday'].sum()} out of {len(rest)} days")
print(f"\nHoliday names:")
print(rest.loc[rest["holiday"] == 1, "holiday_name"].unique())

In [ ]:
# Plot total visitors over time
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rest["total"], linewidth=0.7)
ax.set_title("Total Restaurant Visitors — Daily")
ax.set_ylabel("Total Visitors")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

print("Clear weekly seasonality (7-day cycle) and some holiday spikes.")

In [ ]:
# Highlight holiday vs non-holiday visitor counts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
rest.boxplot(column="total", by="holiday", ax=axes[0])
axes[0].set_title("Visitors by Holiday Status")
axes[0].set_xlabel("Holiday (0=No, 1=Yes)")
axes[0].set_ylabel("Total Visitors")
plt.sca(axes[0])
plt.title("Visitors by Holiday Status")

# Time series with holiday markers
axes[1].plot(rest["total"], linewidth=0.5, alpha=0.6, label="Daily total")
holiday_dates = rest.index[rest["holiday"] == 1]
holiday_vals = rest.loc[rest["holiday"] == 1, "total"]
axes[1].scatter(holiday_dates, holiday_vals, color="red", s=20, zorder=5, label="Holidays")
axes[1].set_title("Visitors with Holidays Highlighted")
axes[1].legend()

plt.tight_layout()
plt.show()

hol_mean = rest.loc[rest["holiday"] == 1, "total"].mean()
nohol_mean = rest.loc[rest["holiday"] == 0, "total"].mean()
print(f"Mean visitors on holidays:     {hol_mean:.1f}")
print(f"Mean visitors on non-holidays: {nohol_mean:.1f}")
print(f"Holiday uplift: {(hol_mean / nohol_mean - 1) * 100:.1f}%")

---
## 2. Train / Test Split

We will hold out the last 60 days for testing.

In [ ]:
HORIZON = 60
train = rest.iloc[:-HORIZON]
test = rest.iloc[-HORIZON:]

print(f"Train: {len(train)} days ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} days ({test.index[0].date()} to {test.index[-1].date()})")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["total"], label="Train", linewidth=0.7)
ax.plot(test["total"], label="Test", color="tab:orange", linewidth=0.7)
ax.axvline(test.index[0], color="grey", linestyle="--", alpha=0.7)
ax.set_title("Train / Test Split — Restaurant Visitors")
ax.set_ylabel("Total Visitors")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Baseline: SARIMA Without Exogenous Variables

We fit a SARIMA model with weekly seasonality ($m=7$) and **no** holiday
information.  This is our baseline to beat.

In [ ]:
# Baseline SARIMA — no exogenous
sarima_model = SARIMAX(
    train["total"],
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_result = sarima_model.fit(disp=False)

print(f"SARIMA(1,0,1)(1,1,1)[7] — no exogenous")
print(f"AIC: {sarima_result.aic:.2f}")
print(sarima_result.summary().tables[1])

In [ ]:
# Forecast baseline
sarima_fc = sarima_result.get_forecast(steps=HORIZON)
sarima_pred = sarima_fc.predicted_mean

sarima_mae = mean_absolute_error(test["total"], sarima_pred)
sarima_rmse = np.sqrt(mean_squared_error(test["total"], sarima_pred))

print(f"Baseline SARIMA (no holidays):")
print(f"  MAE:  {sarima_mae:.2f}")
print(f"  RMSE: {sarima_rmse:.2f}")

---
## 4. SARIMAX: Adding the Holiday Exogenous Variable

Now we add the `holiday` column as an exogenous variable.  The model will
learn that holidays shift visitor counts by some amount $\beta_{\text{holiday}}$.

$$
\text{visitors}_t = \beta_0 + \beta_{\text{holiday}} \cdot \text{holiday}_t + \eta_t
$$

where $\eta_t \sim \text{SARIMA}(1,0,1)(1,1,1)_7$.

In [ ]:
# SARIMAX with holiday exogenous
sarimax_model = SARIMAX(
    endog=train["total"],
    exog=train[["holiday"]],
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_result = sarimax_model.fit(disp=False)

print(f"SARIMAX(1,0,1)(1,1,1)[7] + holiday")
print(f"AIC: {sarimax_result.aic:.2f}")
print(sarimax_result.summary().tables[1])

In [ ]:
print("The 'holiday' coefficient tells us the estimated effect of a holiday")
print("on total restaurant visitors, after accounting for the time series structure.")
print()
print(f"Holiday coefficient: {sarimax_result.params.get('holiday', 'N/A'):.2f}")
print("Interpretation: on a holiday, the model expects this many additional visitors")
print("compared to a non-holiday with the same day-of-week pattern.")

In [ ]:
# Forecast SARIMAX — provide holiday info for the test period
sarimax_fc = sarimax_result.get_forecast(
    steps=HORIZON,
    exog=test[["holiday"]],
)
sarimax_pred = sarimax_fc.predicted_mean
sarimax_ci = sarimax_fc.conf_int(alpha=0.05)

sarimax_mae = mean_absolute_error(test["total"], sarimax_pred)
sarimax_rmse = np.sqrt(mean_squared_error(test["total"], sarimax_pred))

print(f"SARIMAX + holiday:")
print(f"  MAE:  {sarimax_mae:.2f}")
print(f"  RMSE: {sarimax_rmse:.2f}")

---
## 5. Comparison: SARIMA vs SARIMAX

In [ ]:
# Side-by-side comparison
comp = pd.DataFrame({
    "Model": ["SARIMA (no holiday)", "SARIMAX (+ holiday)"],
    "AIC": [sarima_result.aic, sarimax_result.aic],
    "MAE": [sarima_mae, sarimax_mae],
    "RMSE": [sarima_rmse, sarimax_rmse],
}).round(2)

print(comp.to_string(index=False))
print()
improvement_mae = (sarima_mae - sarimax_mae) / sarima_mae * 100
improvement_rmse = (sarima_rmse - sarimax_rmse) / sarima_rmse * 100
print(f"MAE improvement:  {improvement_mae:.1f}%")
print(f"RMSE improvement: {improvement_rmse:.1f}%")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test["total"], label="Actual", color="tab:blue", linewidth=1.5)
ax.plot(sarima_pred, label="SARIMA (no holiday)", color="tab:red", linestyle="--")
ax.plot(sarimax_pred, label="SARIMAX (+ holiday)", color="tab:green", linestyle="--")

# Highlight holidays in the test period
test_holidays = test.index[test["holiday"] == 1]
for hol in test_holidays:
    ax.axvline(hol, color="gold", alpha=0.4, linewidth=3)

ax.set_title("SARIMA vs SARIMAX — Restaurant Visitors")
ax.set_ylabel("Total Visitors")
ax.legend()
plt.tight_layout()
plt.show()

print("Gold vertical bars = holidays in the test period.")
print("The SARIMAX model adjusts its forecast on holiday days.")

In [ ]:
# Zoom in on a holiday period to see the difference clearly
# Find first holiday in test set
if len(test_holidays) > 0:
    hol_date = test_holidays[0]
    window = pd.Timedelta(days=5)
    zoom_mask = (test.index >= hol_date - window) & (test.index <= hol_date + window)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(test.loc[zoom_mask, "total"], marker="o", label="Actual", color="tab:blue")
    ax.plot(sarima_pred[zoom_mask], marker="s", label="SARIMA", color="tab:red", linestyle="--")
    ax.plot(sarimax_pred[zoom_mask], marker="^", label="SARIMAX", color="tab:green", linestyle="--")
    ax.axvline(hol_date, color="gold", alpha=0.5, linewidth=3, label="Holiday")
    ax.set_title(f"Zoomed: Around {hol_date.date()}")
    ax.set_ylabel("Total Visitors")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No holidays in test set to zoom into.")

---
## 6. Residual Diagnostics

In [ ]:
fig = sarimax_result.plot_diagnostics(figsize=(14, 10))
plt.suptitle("SARIMAX Residual Diagnostics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
lb = acorr_ljungbox(sarimax_result.resid.dropna(), lags=[7, 14, 21], return_df=True)
print("Ljung-Box test on SARIMAX residuals:")
print(lb)

---
## 7. Feature Engineering for Exogenous Variables

The holiday dummy is just one type of exogenous variable.  Here are common
feature engineering strategies for time series exogenous regressors.

In [ ]:
# Feature engineering examples

def create_calendar_features(df):
    """Create common calendar-based exogenous features."""
    features = pd.DataFrame(index=df.index)

    # Day of week dummies (6 dummies — drop one to avoid multicollinearity)
    dow = pd.get_dummies(df.index.dayofweek, prefix="dow", drop_first=True, dtype=int)
    dow.index = df.index
    features = pd.concat([features, dow], axis=1)

    # Month-start / month-end indicators
    features["month_start"] = df.index.is_month_start.astype(int)
    features["month_end"] = df.index.is_month_end.astype(int)

    # Trend counter (linear trend)
    features["trend"] = np.arange(len(df))

    return features


cal_features = create_calendar_features(rest)
print("Calendar features created:")
print(cal_features.head(10))
print(f"\nShape: {cal_features.shape}")

### Common Exogenous Variable Types

| Type | Example | Known in advance? |
|------|---------|-------------------|
| **Holiday dummies** | `holiday = 1` on public holidays | Yes (calendar is known) |
| **Day-of-week dummies** | 6 binary columns for Mon-Sat | Yes |
| **Trend counter** | $t = 0, 1, 2, \ldots$ | Yes |
| **Fourier terms** | $\sin(2\pi k t/m)$, $\cos(2\pi k t/m)$ | Yes (next notebook) |
| **Promotional events** | `promo = 1` during a sale | Yes (if planned ahead) |
| **Weather forecasts** | Temperature, precipitation | Approximately (short-term) |
| **Economic indicators** | GDP, interest rates | No (must be forecast separately) |

**Rule of thumb:** the best exogenous variables are **deterministic** — 
you can compute them for any future date without uncertainty.

---
## 8. SARIMAX with Multiple Exogenous: Holiday + Weekday

Let us try adding day-of-week dummies alongside the holiday indicator to
see if we can improve further.

In [ ]:
# Build exog matrix with holiday + day-of-week dummies
dow_dummies = pd.get_dummies(rest.index.dayofweek, prefix="dow", drop_first=True, dtype=int)
dow_dummies.index = rest.index

exog_multi = pd.concat([rest[["holiday"]], dow_dummies], axis=1)

train_exog = exog_multi.iloc[:-HORIZON]
test_exog = exog_multi.iloc[-HORIZON:]

print(f"Exogenous features: {list(exog_multi.columns)}")
print(f"Train exog shape: {train_exog.shape}")
print(f"Test exog shape:  {test_exog.shape}")

In [ ]:
# SARIMAX with holiday + weekday dummies
# Use a non-seasonal ARIMA since day-of-week is now handled by exog
multi_model = SARIMAX(
    endog=train["total"],
    exog=train_exog,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
multi_result = multi_model.fit(disp=False)

multi_fc = multi_result.get_forecast(steps=HORIZON, exog=test_exog)
multi_pred = multi_fc.predicted_mean

multi_mae = mean_absolute_error(test["total"], multi_pred)
multi_rmse = np.sqrt(mean_squared_error(test["total"], multi_pred))

print(f"SARIMAX + holiday + weekday dummies:")
print(f"  AIC:  {multi_result.aic:.2f}")
print(f"  MAE:  {multi_mae:.2f}")
print(f"  RMSE: {multi_rmse:.2f}")

In [ ]:
# Summary comparison of all three models
all_comp = pd.DataFrame({
    "Model": [
        "SARIMA (no exog)",
        "SARIMAX (holiday only)",
        "SARIMAX (holiday + weekday)",
    ],
    "AIC": [sarima_result.aic, sarimax_result.aic, multi_result.aic],
    "MAE": [sarima_mae, sarimax_mae, multi_mae],
    "RMSE": [sarima_rmse, sarimax_rmse, multi_rmse],
}).round(2)

print(all_comp.to_string(index=False))

---
## 9. Application: Alcohol Sales with Trend Exogenous

Let us apply SARIMAX to a different dataset — monthly **Alcohol Sales** — 
using a simple trend counter as an exogenous variable.  This is useful when
the trend component is better modelled by a linear predictor than by
differencing.

In [ ]:
# Load Alcohol Sales
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.columns = ["Sales"]
alcohol.index.freq = "MS"

print(f"Shape: {alcohol.shape}")
print(f"Date range: {alcohol.index[0].date()} to {alcohol.index[-1].date()}")

fig, ax = plt.subplots()
ax.plot(alcohol["Sales"])
ax.set_title("Monthly Alcohol Sales")
ax.set_ylabel("Sales (Millions USD)")
plt.tight_layout()
plt.show()

print("Clear upward trend + strong annual seasonality.")

In [ ]:
# Create trend exogenous and train/test split
alcohol["trend"] = np.arange(len(alcohol))

ALC_HORIZON = 24
alc_train = alcohol.iloc[:-ALC_HORIZON]
alc_test = alcohol.iloc[-ALC_HORIZON:]

print(f"Train: {len(alc_train)} months")
print(f"Test:  {len(alc_test)} months")

In [ ]:
# Baseline: SARIMA on Alcohol Sales
alc_sarima = SARIMAX(
    alc_train["Sales"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

alc_sarima_fc = alc_sarima.get_forecast(steps=ALC_HORIZON)
alc_sarima_pred = alc_sarima_fc.predicted_mean

alc_sarima_mae = mean_absolute_error(alc_test["Sales"], alc_sarima_pred)
alc_sarima_rmse = np.sqrt(mean_squared_error(alc_test["Sales"], alc_sarima_pred))

print(f"SARIMA(1,1,1)(1,1,1)[12] — no exog:")
print(f"  MAE:  {alc_sarima_mae:.2f}")
print(f"  RMSE: {alc_sarima_rmse:.2f}")

In [ ]:
# SARIMAX with trend exogenous
alc_sarimax = SARIMAX(
    endog=alc_train["Sales"],
    exog=alc_train[["trend"]],
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

alc_sarimax_fc = alc_sarimax.get_forecast(
    steps=ALC_HORIZON,
    exog=alc_test[["trend"]],
)
alc_sarimax_pred = alc_sarimax_fc.predicted_mean

alc_sarimax_mae = mean_absolute_error(alc_test["Sales"], alc_sarimax_pred)
alc_sarimax_rmse = np.sqrt(mean_squared_error(alc_test["Sales"], alc_sarimax_pred))

print(f"SARIMAX(1,0,1)(1,1,1)[12] + trend:")
print(f"  MAE:  {alc_sarimax_mae:.2f}")
print(f"  RMSE: {alc_sarimax_rmse:.2f}")

In [ ]:
# Visual comparison for Alcohol Sales
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(alc_test["Sales"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(alc_sarima_pred, label="SARIMA", color="tab:red", linestyle="--")
ax.plot(alc_sarimax_pred, label="SARIMAX + trend", color="tab:green", linestyle="--")
ax.set_title("Alcohol Sales — SARIMA vs SARIMAX")
ax.set_ylabel("Sales (Millions USD)")
ax.legend()
plt.tight_layout()
plt.show()

alc_comp = pd.DataFrame({
    "Model": ["SARIMA (no exog)", "SARIMAX (+ trend)"],
    "MAE": [alc_sarima_mae, alc_sarimax_mae],
    "RMSE": [alc_sarima_rmse, alc_sarimax_rmse],
}).round(2)
print(alc_comp.to_string(index=False))

---
## 10. Guidelines for Exogenous Variables

**When to use exogenous variables:**
- You have external information that is known to affect the target (holidays, promotions, events)
- The external variable is known (or can be reliably forecast) for the forecast horizon
- Adding the variable improves AIC/BIC or out-of-sample accuracy

**When NOT to use exogenous variables:**
- The variable is just a lagged version of the target (use ARIMA instead)
- The variable is not known in advance and difficult to forecast
- The variable does not have a meaningful causal/predictive relationship with the target
- Adding more variables increases AIC or worsens out-of-sample performance (overfitting)

**Practical tips:**
1. Start with a pure SARIMA baseline — always compare
2. Add exogenous variables one at a time and check AIC
3. Verify that the coefficient is statistically significant (p-value < 0.05)
4. Always check residual diagnostics after adding exogenous
5. Prepare your exog matrix for the forecast period *before* calling `get_forecast`

---
## Summary

- **Exogenous variables** add external information to a SARIMA model,
  creating a SARIMAX model that can capture effects the time series
  structure alone would miss.
- The **restaurant visitors** example shows a tangible improvement from
  adding a holiday dummy — the model learns that holidays shift demand.
- **Feature engineering** for exogenous variables includes holiday dummies,
  day-of-week indicators, trend counters, and Fourier terms.
- Exogenous variables must be **known for the forecast period** — this is
  the fundamental constraint.  You cannot use a variable you cannot forecast.
- Always compare against a **pure SARIMA baseline** to confirm that the
  exogenous variables actually help.
- Use AIC/BIC for in-sample comparison and MAE/RMSE on a held-out test set
  for out-of-sample evaluation.

In [ ]:
print("Next notebook: Dynamic harmonic regression — using Fourier terms")
print("as exogenous variables to model complex seasonality.")